Tasks:
1. Data Import and Cleaning:
* Import the dataset using Pandas.
* Clean and preprocess the data, addressing missing values and categorizing data as needed.
* Convert dates and other relevant fields to appropriate formats.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
airplane_crashes_fatalaties_raw = pd.read_csv('Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv', encoding='latin1')
airplane_crashes_fatalaties_raw.head()

In [ ]:
airplane_crashes_fatalaties_raw.shape

In [ ]:
airplane_crashes_fatalaties_raw.describe()

In [ ]:
airplane_crashes_fatalaties_raw.isnull().sum()

In [ ]:
airplane_crashes_fatalaties = airplane_crashes_fatalaties_raw.copy()

# drop flight number because it's missing 73% of the values
airplane_crashes_fatalaties.drop(columns=['Flight #'], inplace=True)

In [ ]:
# fill any aboard and Fatalaties missing values, since these are the fewest missing and they are the totals

# Values are the medians pulled from describe() output
airplane_crashes_fatalaties['Aboard'] = airplane_crashes_fatalaties['Aboard'].fillna(16.0)
airplane_crashes_fatalaties['Fatalities'] = airplane_crashes_fatalaties['Fatalities'].fillna(11.0)

In [ ]:
# fill columns if only one of the Aboard or Fatalaties columns, if we have the other two

# For Aboard columns
mask_crew_ab = airplane_crashes_fatalaties['Aboard Crew'].isnull() & airplane_crashes_fatalaties['Aboard Passangers'].notnull()
airplane_crashes_fatalaties.loc[mask_crew_ab, 'Aboard Crew'] = airplane_crashes_fatalaties['Aboard'] - airplane_crashes_fatalaties['Aboard Passangers']

mask_pass_ab = airplane_crashes_fatalaties['Aboard Passangers'].isnull() & airplane_crashes_fatalaties['Aboard Crew'].notnull()
airplane_crashes_fatalaties.loc[mask_pass_ab, 'Aboard Passangers'] = airplane_crashes_fatalaties['Aboard'] - airplane_crashes_fatalaties['Aboard Crew']

# For Fatalities columns
mask_crew_fat = airplane_crashes_fatalaties['Fatalities Crew'].isnull() & airplane_crashes_fatalaties['Fatalities Passangers'].notnull()
airplane_crashes_fatalaties.loc[mask_crew_fat, 'Fatalities Crew'] = airplane_crashes_fatalaties['Fatalities'] - airplane_crashes_fatalaties['Fatalities Passangers']

mask_pass_fat = airplane_crashes_fatalaties['Fatalities Passangers'].isnull() & airplane_crashes_fatalaties['Fatalities Crew'].notnull()
airplane_crashes_fatalaties.loc[mask_pass_fat, 'Fatalities Passangers'] = airplane_crashes_fatalaties['Fatalities'] - airplane_crashes_fatalaties['Fatalities Crew']

In [ ]:
# Median Crew values from describe(): Aboard Crew = 4.0, Fatalities Crew = 3.0
# Filling Aboard sub-columns
mask_both_ab = airplane_crashes_fatalaties['Aboard Crew'].isnull() & airplane_crashes_fatalaties['Aboard Passangers'].isnull()
airplane_crashes_fatalaties.loc[mask_both_ab, 'Aboard Crew'] = airplane_crashes_fatalaties['Aboard'].apply(lambda x: min(x, 4.0))
airplane_crashes_fatalaties.loc[mask_both_ab, 'Aboard Passangers'] = airplane_crashes_fatalaties['Aboard'] - airplane_crashes_fatalaties['Aboard Crew']

# Filling Fatalities sub-columns
mask_both_fat = airplane_crashes_fatalaties['Fatalities Crew'].isnull() & airplane_crashes_fatalaties['Fatalities Passangers'].isnull()
airplane_crashes_fatalaties.loc[mask_both_fat, 'Fatalities Crew'] = airplane_crashes_fatalaties['Fatalities'].apply(lambda x: min(x, 3.0))
airplane_crashes_fatalaties.loc[mask_both_fat, 'Fatalities Passangers'] = airplane_crashes_fatalaties['Fatalities'] - airplane_crashes_fatalaties['Fatalities Crew']

In [ ]:
airplane_crashes_fatalaties.isnull().sum()

In [ ]:
#convert date column from string to datetime
airplane_crashes_fatalaties['Date'] = pd.to_datetime(airplane_crashes_fatalaties['Date'])


In [ ]:
airplane_crashes_fatalaties['Year'] = airplane_crashes_fatalaties['Date'].dt.year
airplane_crashes_fatalaties['Month'] = airplane_crashes_fatalaties['Date'].dt.month

# Drop the original Date object
airplane_crashes_fatalaties.drop(columns=['Date'], inplace=True)

In [ ]:
# 1. Extract the Hour from the Time string (handling inconsistent formats)
# This keeps only the first two digits (HH) and converts to a numeric format
airplane_crashes_fatalaties['Hour'] = pd.to_numeric(
    airplane_crashes_fatalaties['Time'].str.extract(r'(\d{2}):')[0], 
    errors='coerce'
)

def get_time_slot(hour):
    if pd.isna(hour):
        return 'Unknown'
    elif 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night' # Late night and very early morning

# 3. Apply the categorization
airplane_crashes_fatalaties['Time_Slot'] = airplane_crashes_fatalaties['Hour'].apply(get_time_slot)

# 4. Convert to ML-ready format (One-Hot Encoding)
# This creates separate columns like Time_Slot_Morning, Time_Slot_Unknown, etc.
time_dummies = pd.get_dummies(airplane_crashes_fatalaties['Time_Slot'], prefix='Time_Slot')
airplane_crashes_fatalaties = pd.concat([airplane_crashes_fatalaties, time_dummies], axis=1)


In [ ]:
# Drops rows only if they are missing values in these specific columns, since its so few total rows
airplane_crashes_fatalaties.dropna(subset=['Location', 'Operator', 'AC Type'], inplace=True)

In [ ]:
# Shows the unique values and how often they appear
print(airplane_crashes_fatalaties['Route'].value_counts().head(20))

# Shows the total number of unique categories
print(f"\nTotal Unique Routes: {airplane_crashes_fatalaties['Route'].nunique()}")

In [ ]:
# Define common purpose keywords found in your data
purpose_keywords = ['Training', 'Test', 'Sightseeing', 'Demonstration', 'Survey', 'Military', 'Air show']

def extract_purpose(route):
    if pd.isna(route):
        return 'Unknown'
    route_str = str(route)
    for word in purpose_keywords:
        if word.lower() in route_str.lower():
            return word
    return 'Commercial/Scheduled' # Default for actual city-to-city routes

airplane_crashes_fatalaties['Flight_Purpose'] = airplane_crashes_fatalaties['Route'].apply(extract_purpose)

# Create the binary 'Point-to-Point' flag we discussed
airplane_crashes_fatalaties['Is_Point_To_Point'] = airplane_crashes_fatalaties['Route'].str.contains(' - ').fillna(0).astype(int)

# One-Hot Encode the Flight_Purpose (just like you did for Time_Slot)
purpose_dummies = pd.get_dummies(airplane_crashes_fatalaties['Flight_Purpose'], prefix='Purpose')
airplane_crashes_fatalaties = pd.concat([airplane_crashes_fatalaties, purpose_dummies], axis=1)


In [ ]:
# List of columns that are now redundant or unreadable by ML
cols_to_drop = ['Time', 'Hour', 'Route', 'Flight_Purpose', 'Registration', 'cn/ln', 'Summary']

# Drop them all at once (check if they exist first to avoid errors)
airplane_crashes_fatalaties.drop(columns=[c for c in cols_to_drop if c in airplane_crashes_fatalaties.columns], inplace=True)

In [ ]:
airplane_crashes_fatalaties.isnull().sum()

In [ ]:
# Shows the unique values and how often they appear
print(airplane_crashes_fatalaties['Ground'].value_counts().head(20))

# Shows the total number of unique categories
print(f"\nTotal Ground info: {airplane_crashes_fatalaties['Ground'].nunique()}")

In [ ]:
# Filling the remaining 38 gaps with 0.0
airplane_crashes_fatalaties['Ground'] = airplane_crashes_fatalaties['Ground'].fillna(0.0)

In [ ]:
cols_to_check = ['Location', 'Operator', 'AC Type']

for col in cols_to_check:
    unique_count = airplane_crashes_fatalaties[col].nunique()
    print(f"{col} has {unique_count} unique values.")

In [ ]:
# List of high-cardinality categorical columns
high_card_cols = ['Location', 'Operator', 'AC Type']

for col in high_card_cols:
    # 1. Calculate the frequency (count) of each value
    freq_map = airplane_crashes_fatalaties[col].value_counts()
    
    # 2. Map those counts back to the original column
    airplane_crashes_fatalaties[f'{col}_Freq'] = airplane_crashes_fatalaties[col].map(freq_map)

# 3. Now drop the original string columns so the model only sees the numbers
airplane_crashes_fatalaties.drop(columns=high_card_cols, inplace=True)


In [ ]:
airplane_crashes_fatalaties.info()

In [ ]:
# Select all boolean columns and convert them to integers
bool_cols = airplane_crashes_fatalaties.select_dtypes(include='bool').columns
airplane_crashes_fatalaties[bool_cols] = airplane_crashes_fatalaties[bool_cols].astype(int)


In [ ]:
# Drop any remaining string/object columns
airplane_crashes_fatalaties.drop(columns=['Time_Slot'], inplace=True, errors='ignore')

In [ ]:
airplane_crashes_fatalaties.dtypes

In [ ]:
# Check how many unique frequency levels exist
print(airplane_crashes_fatalaties[['Location_Freq', 'Operator_Freq', 'AC Type_Freq']].nunique())

# Peek at the top of the data to see the numbers
print(airplane_crashes_fatalaties[['Location_Freq', 'Operator_Freq', 'AC Type_Freq']].head())


In [ ]:
airplane_crashes_fatalaties.reset_index(drop=True, inplace=True)

In [ ]:
# Should return False
print(airplane_crashes_fatalaties.isnull().values.any())

# Should show only float, int64, or int32 (no objects/strings)
print(airplane_crashes_fatalaties.dtypes.unique())


2. Exploratory Data Analysis:
* Use Pandas to explore basic statistics such as the number of crashes, fatalities, and survival rates.
* Analyze the frequency of crashes over time to identify any trends.

In [ ]:
# Total counts
total_crashes = len(airplane_crashes_fatalaties)
total_fatalities = airplane_crashes_fatalaties['Fatalities'].sum()
total_aboard = airplane_crashes_fatalaties['Aboard'].sum()

# Overall Survival Rate
overall_survival_rate = 1 - (total_fatalities / total_aboard)

print(f"Total Crashes: {total_crashes}")
print(f"Total Fatalities: {total_fatalities:,.0f}")
print(f"Overall Survival Rate: {overall_survival_rate:.2%}")

In [ ]:
airplane_crashes_fatalaties['Survival_Rate'] = np.where(
    airplane_crashes_fatalaties['Aboard'] > 0, 
    1 - (airplane_crashes_fatalaties['Fatalities'] / airplane_crashes_fatalaties['Aboard']), 
    0
)

In [ ]:
# Grouping by year to see counts and fatality averages
yearly_stats = airplane_crashes_fatalaties.groupby('Year').agg({
    'Year': 'count',                 # Number of crashes
    'Fatalities': 'sum',             # Total deaths per year
    'Survival_Rate': 'mean'          # Average survival rate per year
}).rename(columns={'Year': 'Crash_Count'})

# Display the most recent 10 years in the dataset
print(yearly_stats.tail(10))

In [ ]:
print(airplane_crashes_fatalaties[['Aboard', 'Fatalities', 'Survival_Rate']].describe())

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12, 6))

# Axis 1: Number of Crashes
ax1.set_xlabel('Year')
ax1.set_ylabel('Number of Crashes', color='tab:blue')
ax1.plot(yearly_stats.index, yearly_stats['Crash_Count'], color='tab:blue', label='Crashes')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Axis 2: Survival Rate
ax2 = ax1.twinx()
ax2.set_ylabel('Average Survival Rate', color='tab:red')
ax2.plot(yearly_stats.index, yearly_stats['Survival_Rate'], color='tab:red', linestyle='--', label='Survival Rate')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Crash Frequency vs. Survival Rate Over Time')
fig.tight_layout()
plt.show()


3. Statistical Analysis:
* Apply SciPy to analyze the distribution of fatalities and survival rates. Calculate key statistics like mean, median, and standard deviation.
* Conduct a hypothesis test (e.g., comparing the average number of fatalities in different decades or regions).

In [ ]:
from scipy import stats

# Calculate stats for Fatalities
f_mean = np.mean(airplane_crashes_fatalaties['Fatalities'])
f_median = np.median(airplane_crashes_fatalaties['Fatalities'])
f_std = np.std(airplane_crashes_fatalaties['Fatalities'])
f_skew = stats.skew(airplane_crashes_fatalaties['Fatalities'])

print(f"Fatalities -> Mean: {f_mean:.2f}, Median: {f_median}, Std Dev: {f_std:.2f}, Skew: {f_skew:.2f}")

In [ ]:
# Create two groups for comparison
group_70s = airplane_crashes_fatalaties[airplane_crashes_fatalaties['Year'].between(1970, 1979)]['Fatalities']
group_00s = airplane_crashes_fatalaties[airplane_crashes_fatalaties['Year'].between(2000, 2009)]['Fatalities']

# Run the test
u_stat, p_value = stats.mannwhitneyu(group_70s, group_00s)

print(f"Mann-Whitney U Test p-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Significant difference found between the two decades.")
else:
    print("Result: No significant difference found.")


In [ ]:
# Spearman correlation between Year and Survival_Rate
corr, p_val = stats.spearmanr(airplane_crashes_fatalaties['Year'], airplane_crashes_fatalaties['Survival_Rate'])

print(f"Correlation Coefficient: {corr:.4f}")
print(f"P-value: {p_val:.4g}")

4. Visualization:
* Create charts and graphs using Matplotlib and Seaborn to visualize the findings from your exploratory data analysis and statistical tests.
* Examples might include time series plots of crashes over years, bar charts of crashes by region, and histograms of fatalities.

In [ ]:
import seaborn as sns

# Set the visual style
sns.set_theme(style="whitegrid")

plt.figure(figsize=(14, 6))
yearly_stats = airplane_crashes_fatalaties.groupby('Year').agg({'Aboard': 'count', 'Fatalities': 'sum'})

sns.lineplot(data=yearly_stats, x=yearly_stats.index, y='Aboard', label='Total Crashes', color='blue', linewidth=2)
sns.lineplot(data=yearly_stats, x=yearly_stats.index, y='Fatalities', label='Total Fatalities', color='red', alpha=0.6)

plt.title('Historical Trend: Annual Crashes vs. Fatalities', fontsize=15)
plt.ylabel('Count')
plt.xlabel('Year')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
# Clipping at 150 to see the distribution better (ignoring the extreme outliers for the visual)
sns.histplot(airplane_crashes_fatalaties['Fatalities'], bins=40, kde=True, color='teal')

plt.xlim(0, 150) # Focus on where 95% of the data lives
plt.title('Distribution of Fatalities per Crash (Truncated at 150)', fontsize=14)
plt.xlabel('Number of Fatalities')
plt.ylabel('Frequency')
plt.axvline(airplane_crashes_fatalaties['Fatalities'].median(), color='red', linestyle='--', label=f'Median: {airplane_crashes_fatalaties["Fatalities"].median()}')
plt.legend()
plt.show()


In [ ]:
# 1. Identify which columns are the 'Purpose' dummies
purpose_cols = [col for col in airplane_crashes_fatalaties.columns if col.startswith('Purpose_')]

# 2. Create a temporary column that tells us the 'max' purpose for each row
# This finds the column where the value is 1 and strips the 'Purpose_' prefix
airplane_crashes_fatalaties['Temp_Purpose'] = airplane_crashes_fatalaties[purpose_cols].idxmax(axis=1).str.replace('Purpose_', '')

# 3. Now run the plot code using 'Temp_Purpose'
plt.figure(figsize=(12, 6))
purpose_order = airplane_crashes_fatalaties.groupby('Temp_Purpose')['Fatalities'].mean().sort_values(ascending=False).index

sns.barplot(data=airplane_crashes_fatalaties, x='Temp_Purpose', y='Fatalities', palette='viridis', order=purpose_order)

plt.title('Average Fatalities by Flight Purpose', fontsize=14)
plt.xticks(rotation=45)
plt.show()

# 4. Clean up: Drop the temporary column so it doesn't mess up your ML model later
airplane_crashes_fatalaties.drop(columns=['Temp_Purpose'], inplace=True)

In [ ]:
# 1. Identify only the 'Purpose' dummy columns
purpose_cols = [col for col in airplane_crashes_fatalaties.columns if col.startswith('Purpose_')]

# 2. Sum each column to get total counts per purpose
crash_counts = airplane_crashes_fatalaties[purpose_cols].sum().sort_values(ascending=False)

# 3. Clean up labels for the plot (remove the 'Purpose_' prefix)
crash_counts.index = crash_counts.index.str.replace('Purpose_', '')


In [ ]:
plt.figure(figsize=(12, 6))

# Plotting the frequencies
sns.barplot(x=crash_counts.index, y=crash_counts.values, palette='magma')

plt.title('Total Number of Crashes by Flight Purpose', fontsize=15)
plt.ylabel('Number of Crashes')
plt.xlabel('Flight Purpose')
plt.xticks(rotation=45)

# Adding the exact count on top of each bar for clarity
for i, v in enumerate(crash_counts.values):
    plt.text(i, v + 5, str(int(v)), ha='center', fontweight='bold')

plt.show()

5. Insight and Report:
* Summarize your findings and provide insights into the patterns or anomalies discovered in the data.
* Prepare a well-structured report including all code, visualizations, and interpretations.

After cleaning and analyzing 4,972 airplane crashes spanning over a century, several clear patterns and statistical anomalies have emerged. Here is the summary of the data's "story":

1. The "Safety Paradox"
While the number of crashes peaked significantly between 1945 and 1970 (the rapid expansion of air travel), the Survival Rate has shown a statistically significant—though weak—improvement over time (
p=0.0001). This suggests that while we haven't eliminated crashes, we have made them significantly more survivable through better engineering and safety protocols.

2. Extreme Skewness (The "Long Tail" Risk)
The data is highly skewed (4.60).
The Reality: The median number of fatalities is only 11, meaning half of all recorded crashes involve relatively few deaths.
The Anomaly: The mean is much higher (22.42) because of "Black Swan" events—massive disasters with hundreds of fatalities that pull the average up. This confirms that aviation safety is a battle against rare, high-impact outliers rather than frequent small ones.

3. Flight Purpose as a Risk Predictor
By engineering the Flight_Purpose feature, we discovered that Risk isn't distributed equally:
Commercial/Scheduled flights account for the highest volume of crashes, but this is due to their massive global flight frequency.
Test and Training flights show a distinct pattern of "high frequency, lower lethality." These are the "fender benders" of the sky—accidents happen more often during these maneuvers, but because fewer people are aboard, the total fatality count per incident is lower.

4. Operational "Hotspots"
Through Frequency Encoding, we identified that certain Operators and Aircraft Types appear in the data far more often than others.
High-frequency operators (major national carriers) show different survival patterns than low-frequency operators (private/charter), likely due to differences in maintenance standards and pilot training requirements.

5. Time and Visibility
The Time_Slot analysis (engineered from the messy raw time data) indicates that visibility and fatigue are underlying factors. Crashes categorized in the Night or Unknown (often older, less documented) slots tend to show different fatality distributions than Afternoon flights, where visibility is highest.

Final Insight for Modeling
Because the correlation between single variables (like Year) and Survival is weak (0.0538), a simple linear approach won't work for prediction. The true "signal" in this data lies in the interaction between features—for example, the combination of a specific Aircraft Type, a Night time slot, and a Test Flight purpose likely creates a high-risk profile that a Machine Learning model can now identify.